In [ ]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Project-wide publication styling helpers.
sys.path.insert(0, str(Path("../../meta/tools").resolve()))
import plot_utils as pu  # noqa: E402

pu.set_publication_style()

SAVEPATH = "./figures"
os.makedirs(SAVEPATH, exist_ok=True)

RESULTS_PATH = Path('../../.local/bench-results/consolidated_results.json')
with RESULTS_PATH.open() as f:
    data = json.load(f)

exp = data['experiments']['multi_run_7host_variance']
hosts = exp['hosts']
matrices = exp['matrices']
print('hosts:', hosts)
print('matrices:', list(matrices.keys()))


In [ ]:
def to_matrix(stats, hosts, key='mean'):
    n = len(hosts)
    M = np.full((n, n), np.nan)
    for i, src in enumerate(hosts):
        for j, dst in enumerate(hosts):
            cell = stats.get(src, {}).get(dst)
            if cell is None:
                continue
            v = cell.get(key) if isinstance(cell, dict) else cell
            if v is None:
                continue
            M[i, j] = float(v)
    return M

latency = to_matrix(matrices['http_latency_avg_ms'], hosts)
throughput = to_matrix(matrices['throughput_mbps'], hosts)

# The multi-host bench deliberately NULLs the euler<->sgs-amd-01 cells (see
# experiment description: a different effective path applies in the multi-host
# context). The canonical numbers for that pair live in the
# `direct_dial_only_pair` experiment — overlay them so the heatmap renders.
ddop = data['experiments'].get('direct_dial_only_pair') or {}
ddop_metrics = ddop.get('metrics', {})
for primary, src_key in (('http_latency_avg_ms', 'http_latency_avg_ms'),
                         ('throughput_mbps', 'throughput_mbps')):
    target_mat = latency if primary == 'http_latency_avg_ms' else throughput
    src_cells = ddop_metrics.get(src_key, {})
    for pair_str, cell in src_cells.items():
        if '->' not in pair_str:
            continue
        src, dst = pair_str.split('->')
        if src not in hosts or dst not in hosts:
            continue
        v = cell.get('mean') if isinstance(cell, dict) else cell
        if v is None:
            continue
        target_mat[hosts.index(src), hosts.index(dst)] = float(v)


raw = data['experiments']['raw_baseline']
ping_raw = to_matrix(raw['ping_avg_ms'], hosts)
iperf_raw = to_matrix(raw['iperf3_mbps'], hosts, key='sent_mbps_mean')

# Overlay raw ping from contrib/network-profiler's JSONL output. Each line is
# one measurement; we keep the latest ok=True entry per (source, target) and
# prefer it over the consolidated raw_baseline cell (more recent, and fills
# in pairs the consolidated bench didn't cover, e.g. sgs-amd-01 <-> euler).
JSONL_PATH = Path('../../.local/results/network.jsonl')
if JSONL_PATH.exists():
    latest = {}
    with JSONL_PATH.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            if rec.get('kind') != 'ping' or not rec.get('ok'):
                continue
            avg = rec.get('metrics', {}).get('avg')
            if avg is None:
                continue
            key = (rec['source'], rec['target'])
            prev = latest.get(key)
            if prev is None or rec['timestamp'] > prev['timestamp']:
                latest[key] = {'avg': float(avg), 'timestamp': rec['timestamp']}
    overlaid = 0
    for (src, dst), cell in latest.items():
        if src not in hosts or dst not in hosts:
            continue
        i, j = hosts.index(src), hosts.index(dst)
        ping_raw[i, j] = cell['avg']
        overlaid += 1
    print(f'overlaid {overlaid} ping cells from {JSONL_PATH.name}')
else:
    print(f'no JSONL overlay (missing {JSONL_PATH})')

off_diag = latency.size - len(hosts)
print('latency (ms) range:', np.nanmin(latency), '-', np.nanmax(latency))
print('throughput (Mbps) range:', np.nanmin(throughput), '-', np.nanmax(throughput))
print(f'raw ping coverage: {int(np.isfinite(ping_raw).sum())} / {off_diag}')
print(f'raw iperf3 coverage: {int(np.isfinite(iperf_raw).sum())} / {off_diag}')

In [ ]:
DISPLAY_NAME = {
    'bristen': 'Bristen',
    'clariden': 'Clariden',
    'euler': 'Euler',
    'jsc': 'JSC',
    'oci-1': 'OCI-1',
    'oci-2': 'OCI-2',
    'sgs-amd-01': 'ETH-local',
}
# Display order: JSC last (high-latency outlier).
DISPLAY_ORDER = ['bristen', 'clariden', 'euler', 'oci-1', 'oci-2', 'sgs-amd-01', 'jsc']
_order_idx = [hosts.index(h) for h in DISPLAY_ORDER]
display_labels = [DISPLAY_NAME[h] for h in DISPLAY_ORDER]
# Reorder into new arrays so re-running this cell stays idempotent.
latency_ord = latency[np.ix_(_order_idx, _order_idx)]
throughput_ord = throughput[np.ix_(_order_idx, _order_idx)]
ping_raw_ord = ping_raw[np.ix_(_order_idx, _order_idx)]
iperf_raw_ord = iperf_raw[np.ix_(_order_idx, _order_idx)]


def annotate(ax, M, M_raw=None, fmt='{:.0f}', raw_fmt='{:.0f}', dark_threshold=None):
    """Top line: M value (color flips white above dark_threshold). Second line: raw value if available."""
    def _fmt(value, spec):
        # Positive values that round to "0" are below the format's resolution —
        # show "<1" so readers don't think the link is free.
        s = spec.format(value)
        if value > 0 and float(s) == 0:
            return '<1'
        return s

    n = M.shape[0]
    if dark_threshold is None:
        dark_threshold = np.nanmean(M)
    for i in range(n):
        for j in range(n):
            v = M[i, j]
            has_raw = M_raw is not None and np.isfinite(M_raw[i, j])
            if np.isnan(v):
                if has_raw:
                    # No primary measurement but raw ping/iperf is available —
                    # render plainly so it reads like any other cell.
                    ax.text(
                        j, i, _fmt(M_raw[i, j], raw_fmt),
                        ha='center', va='center', color='black', fontsize=11,
                    )
                else:
                    ax.text(j, i, '—', ha='center', va='center', color='#999999', fontsize=14)
                continue
            color = 'white' if v >= dark_threshold else 'black'
            if has_raw:
                ax.text(j, i, _fmt(v, fmt), ha='center', va='bottom', color=color, fontsize=11)
                ax.text(
                    j, i, '(' + _fmt(M_raw[i, j], raw_fmt) + ')',
                    ha='center', va='top', color=color, fontsize=9, alpha=0.85,
                )
            else:
                ax.text(j, i, _fmt(v, fmt), ha='center', va='center', color=color, fontsize=11)


def set_panel_title(ax, main, sub):
    """Two-line title: bold main on top, smaller dim subtitle below."""
    ax.set_title(main, fontsize=15, fontweight='bold', pad=18)
    ax.text(
        0.5, 1.01, sub,
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=11, color='#555555',
    )


fig, axes = plt.subplots(
    nrows=1, ncols=2, figsize=(9.0, 4.25),
    gridspec_kw={'width_ratios': [1.0, 1.0], 'wspace': 0.05},
    constrained_layout=True,
)
fig.get_layout_engine().set(w_pad=0.01, wspace=0.05)
ax_lat, ax_thr = axes

# With aspect='equal' the axes box shrinks to a square and is centered in its
# grid cell by default — that centering is the leftover whitespace between panels.
# Anchor each panel toward the gap so the two heatmaps sit close together.
ax_lat.set_anchor('E')  # left panel hugs the right edge of its cell
ax_thr.set_anchor('W')  # right panel hugs the left edge of its cell

# --- latency heatmap (HTTP avg, ms) with raw ICMP ping in second line ---
im_lat = ax_lat.imshow(
    np.ma.masked_invalid(latency_ord),
    cmap='magma_r',
    norm=LogNorm(vmin=max(np.nanmin(latency_ord), 1.0), vmax=np.nanmax(latency_ord)),
    aspect='equal',
)
set_panel_title(ax_lat, 'Latency (ms)', '')
ax_lat.set_xticks(range(len(display_labels)))
ax_lat.set_yticks(range(len(display_labels)))
ax_lat.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=12)
ax_lat.set_yticklabels(display_labels, fontsize=12)
ax_lat.set_xlabel('destination', fontsize=13)
ax_lat.set_ylabel('source', fontsize=13)
annotate(ax_lat, latency_ord, M_raw=ping_raw_ord, fmt='{:.0f}', raw_fmt='{:.0f}',
         dark_threshold=np.nanpercentile(latency_ord, 65))
cb_lat = fig.colorbar(im_lat, ax=ax_lat, fraction=0.046, pad=0.02)
cb_lat.set_label('ms (log scale)', fontsize=12)
cb_lat.ax.tick_params(labelsize=10)

# --- throughput heatmap (libp2p stream, Mbps) with raw iperf3 in second line ---
# YlGn: light yellow at low values (legible black text), dark green at high (legible white).
# Most cells live in the 40-70 Mbps range; the two oci-1<->oci-2 outliers (~360 Mbps)
# saturate the dark end without crushing the rest of the scale into a dark band.
thr_vmin = float(np.nanmin(throughput_ord))
thr_vmax = float(np.nanmax(throughput_ord))
im_thr = ax_thr.imshow(
    np.ma.masked_invalid(throughput_ord),
    cmap='YlGn',
    vmin=thr_vmin,
    vmax=thr_vmax,
    aspect='equal',
)
set_panel_title(ax_thr, 'Throughput (Mbps)', '')
ax_thr.set_xticks(range(len(display_labels)))
ax_thr.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=12)
# Drop y-axis label and tick labels — rows align with the left panel, which already names them.
ax_thr.set_yticks(range(len(display_labels)))
ax_thr.set_yticklabels([])
ax_thr.set_xlabel('destination', fontsize=13)
# Switch text to white only on the dark end of the YlGn scale (top ~25%).
annotate(ax_thr, throughput_ord, M_raw=iperf_raw_ord, fmt='{:.0f}', raw_fmt='{:.0f}',
         dark_threshold=thr_vmin + 0.75 * (thr_vmax - thr_vmin))
cb_thr = fig.colorbar(im_thr, ax=ax_thr, fraction=0.046, pad=0.02)
cb_thr.set_label('Mbps', fontsize=12)
cb_thr.ax.tick_params(labelsize=10)
pu.save_figure(fig, f"{SAVEPATH}/network_profiler", formats=['pdf'])
plt.show()